In [19]:
import numpy as np
import random 

class Quadratic ():
    def __init__ (self, dim, flow_matrix, dist_matrix, pop_size, no_of_iterations, Q , rho , beta , alpha):
        #initialize all parameters to be used later 
        self.__flow_matrix = flow_matrix
        self.__dist_matrix = dist_matrix
        self.__facilities = [fac for fac in range(dim)] # to give a names to each facility
        self.__dim = dim # number of facilities/dimensions of the problem 
        self.__intialize_parameters( pop_size , no_of_iterations, Q , rho , beta , alpha )

    #_______________________________________________
    def __intialize_parameters(self, pop_size , no_of_iterations, Q , rho , beta , alpha ):
        #To intialize the HYPERPARAMETERS of the algorithm 
        self.__pop_size = pop_size
        self.__no_of_iterations = no_of_iterations 
        self.__Q = Q
        self.__rho = rho
        self.__beta = beta 
        self.__alpha = alpha 
        self.__pheromone = np.full((self.__dim , self.__dim) , fill_value = 1.0) # intialize the pheromone map 

    #_______________________________________________
    def __stating_ants(self):
        #state each ant in the first postion (pick any intialize FACILITY randomly)
        population = []
        one_ant = ['']
        for ant in range (self.__pop_size):
            ant = one_ant.copy()
            ant[0] = random.choice(self.__facilities) # pick one of the facilities randomly 
            population.append(ant)

        return population

    #_______________________________________________
    def __transition (self, ant):
        
        probabilities = np.zeros(self.__dim)

        for facility in self.__facilities : # try all facilities to pick next 
            cost = 0 
            if facility in ant : 
                continue 
            for loc in range (len(ant)) : 
                cost += self.__dist_matrix[loc][len(ant)] * self.__flow_matrix[ant[loc]][facility]

            tau = self.__pheromone[ant[-1],facility] ** self.__alpha 
            cost = (1 / (cost + 1)) ** self.__beta 
            probabilities[facility] = tau * cost

        sum_of_prob = sum(probabilities)
        probabilities /= sum_of_prob
        next_state = [np.random.choice(self.__facilities, size = 1 , p = probabilities).item()]
                     
        return next_state

     #_______________________________________________
    def __calculate_fitness(self, ant):
        cost = 0 
        for curr_loc in range (len(ant)):
            for anoth_loc in range (len(ant)):
                #∑i ∑j Dist(i, j) x Flow(fi, fj)
                cost += self.__dist_matrix[curr_loc][anoth_loc] * self.__flow_matrix[ant[curr_loc]][ant[anoth_loc]]

        return cost

   #_______________________________________________
    def __update_pheromone(self, best_ant, fitness, count):
        self.__pheromone *= (1 - self.__rho) 
        for inx in range (len(best_ant) - 1 ):
            self.__pheromone[best_ant[inx], best_ant[inx + 1 ]] += (self.__Q / fitness)*count

    #_______________________________________________
    def solution (self):
        # Putting all together
        #save settings during execution 
        best_fitness_overall = None
        best_solution = None 

        # Perform iterations of Ant Colony Optimization
        for iteration in range (self.__no_of_iterations):
            population = self.__stating_ants() 

            for ant in population :
                # For each ant , it will add n positions to its solution 
                for pos in range (self.__dim - 1):
                    ant.extend(self.__transition(ant))

            fitness_values = []
            for ant in population:
                fitness_values.append(self.__calculate_fitness(ant))

            best_index = fitness_values.index(min(fitness_values))
            best_ant = population[best_index]
            best_fitness = fitness_values[best_index]
            best_counts = population.count(best_ant)

            self.__update_pheromone(best_ant , best_fitness , best_counts)

            if best_fitness_overall is None or best_fitness < best_fitness_overall :
                best_fitness_overall = best_fitness
                best_solution = population [best_index]
                print (f"\rA solution found in iteration : {iteration} with fitness = {best_fitness_overall}")


        self.__population = population
        return best_solution , best_fitness_overall

     #_______________________________________________
        return self.__population, self.__pheromone
        

In [20]:
flow_matrix = np.array([
    [0, 11, 3, 5, 1],
    [11, 0, 7, 6, 5],
    [3, 7, 0, 2, 8],
    [5, 6, 2, 0, 1],
    [1, 5, 8, 1, 0]
])
dist_matrix = np.array([
    [ 0,  3,  5,  3,  2],   # Loc0
    [ 3,  0, 25,  1,  8],   # Loc1
    [ 5, 25,  0,  13,  9],   # Loc2
    [ 3,  1,  13,  0,  7],   # Loc3
    [ 2,  8,  9,  7,  0]    # Loc4
])
quadratic = Quadratic( dim = 5 , flow_matrix = flow_matrix , dist_matrix = dist_matrix , pop_size = 30, no_of_iterations = 2000 , Q = 3 , rho = 0.26 , beta = 0.8 , alpha = 1.2)
sol = quadratic.solution()
assignment = [int(x) for x in sol[0]]
print(f"The optimal Solution is ({assignment}) with Total cost = ({sol[1]})")

A solution found in iteration : 0 with fitness = 470
A solution found in iteration : 1 with fitness = 442
The optimal Solution is ([1, 4, 3, 2, 0]) with Total cost = (442)


In [22]:
quad.get_conf()

([[1, 4, 3, 2, 0],
  [4, 3, 2, 0, 1],
  [1, 4, 3, 2, 0],
  [3, 2, 0, 4, 1],
  [2, 0, 4, 3, 1],
  [0, 4, 3, 2, 1],
  [0, 4, 3, 2, 1],
  [4, 3, 2, 0, 1],
  [0, 4, 3, 2, 1],
  [4, 3, 2, 0, 1],
  [4, 3, 2, 0, 1],
  [4, 3, 2, 0, 1],
  [2, 0, 4, 3, 1],
  [2, 0, 4, 3, 1],
  [1, 4, 3, 2, 0],
  [0, 4, 3, 2, 1],
  [2, 0, 4, 3, 1],
  [2, 0, 4, 3, 1],
  [1, 4, 3, 2, 0],
  [1, 4, 3, 2, 0],
  [2, 0, 4, 3, 1],
  [1, 4, 3, 2, 0],
  [0, 4, 3, 2, 1],
  [1, 4, 3, 2, 0],
  [1, 4, 3, 2, 0],
  [1, 4, 3, 2, 0],
  [2, 0, 4, 3, 1],
  [3, 2, 0, 4, 1],
  [0, 4, 3, 2, 1],
  [1, 4, 3, 2, 0]],
 array([[2.90696272e-262, 2.90696272e-262, 2.90696272e-262,
         2.90696272e-262, 7.01582826e-138],
        [3.01977608e-262, 2.90696272e-262, 3.25358578e-262,
         2.90696272e-262, 1.66766671e-001],
        [1.66766671e-001, 7.01582826e-138, 2.90696272e-262,
         3.25358578e-262, 2.90696272e-262],
        [2.90696272e-262, 2.90696272e-262, 1.66766671e-001,
         2.90696272e-262, 3.25358578e-262],
        [3.25